# 03. Detailed BattINFO Flow: Coin-Cell Descriptor to Publication

This notebook demonstrates the richer BattINFO authoring path.

You will:

- author a detailed `CellSpecification` for a coin cell
- save the descriptor record
- derive a canonical `CellType`
- link it to a cell, test, and dataset
- build a publication package that includes the detailed specification

Everything writes under `.battinfo/notebooks/03-detailed-coin-cell`.


In [ ]:
from pathlib import Path

from battinfo import (
    Workspace,
    bom,
    construction,
    derive_cell_type,
    electrode,
    electrolyte_recipe,
    material,
    properties,
    quantity,
    separator_spec,
    source,
)


In [ ]:
workspace = Workspace(root=Path('.battinfo/notebooks/03-detailed-coin-cell'), clean=True)

dataset_file = workspace.root / 'inputs' / 'coin-cell-capacity.csv'
dataset_file.parent.mkdir(parents=True, exist_ok=True)
dataset_file.write_text(
    'time_s,voltage_v\n0,3.20\n60,3.08\n120,2.97\n',
    encoding='utf-8',
)

dataset_file


## 1. Describe a detailed coin cell

This uses the richer authoring helpers for electrodes, electrolyte, separator, construction, and explicit quantitative properties.


In [ ]:
positive_electrode = electrode(
    bom=bom(
        active_material=material('MnO2', comment='Electrolytic manganese dioxide cathode powder.'),
        additive='Carbon black',
        binder='PTFE',
    ),
    loading=quantity(22.0, 'mg/cm^2'),
    current_collector='Nickel-plated steel can',
    coating_comment='Cathode mix pressed into the positive can.',
    comment='Representative cathode construction for a coin-cell demonstration.',
)

negative_electrode = electrode(
    bom=bom(active_material=material('Lithium metal foil')),
    loading=quantity(8.0, 'mg/cm^2'),
    current_collector='Nickel-plated steel cap',
    coating_comment='Lithium foil laminated to the negative side.',
    comment='Representative lithium-metal negative electrode for a primary coin cell.',
)

electrolyte = electrolyte_recipe(
    family='organic',
    solvents=['Propylene carbonate', '1,2-dimethoxyethane'],
    salt='LiClO4',
    salt_concentration=quantity(1.0, 'mol/L'),
    additives=['VC'],
    comment='Simple organic electrolyte example for the notebook walkthrough.',
)

separator = separator_spec(
    material='Microporous polypropylene separator',
    thickness=quantity(25, 'um'),
    comment='Single separator disk cut for the coin cell.',
)

specification = workspace.describe_cell(
    manufacturer='DemoCellCo',
    model='CR2032-DETAILED',
    format='coin',
    chemistry='Li/MnO2',
    size_code='R2032',
    positive_electrode_basis='MnO2',
    negative_electrode_basis='Li-metal',
    positive_electrode=positive_electrode,
    negative_electrode=negative_electrode,
    electrolyte=electrolyte,
    separator=separator,
    construction=construction(
        assembly_type='stacked',
        layering='single_layer',
        layer_count=3,
        comment='Positive can, separator, and lithium negative stack.',
    ),
    properties=properties(
        nominal_voltage=quantity(3.0, 'V'),
        nominal_capacity=quantity(0.23, 'Ah'),
        diameter=quantity(20.0, 'mm'),
        height=quantity(3.2, 'mm'),
        mass=quantity(3.1, 'g'),
    ),
    source=source(
        type='datasheet',
        name='Coin-cell specification sheet',
        file='coin-cell-specification-sheet.md',
        retrieved_at=1773950000,
        comment='Human-authored detailed source document used as the descriptor basis.',
    ),
    specification_comment='Detailed coin-cell descriptor for BattINFO notebook 03.',
    comment='Demonstrates the rich descriptor path before publication packaging.',
)

{'specification_id': specification.id, 'model': specification.model, 'format': specification.format}


## 2. Save the detailed descriptor


In [ ]:
description_result = workspace.save_descriptions()

{
    'entry_count': description_result['rdf']['entry_count'],
    'description_output_root': description_result['rdf']['out_dir'],
}


## 3. Derive a canonical cell type and link it into the standard record chain

The detailed descriptor is richer than the canonical `CellType`. `derive_cell_type(...)` gives you the simpler published product model for the normal `CellType -> Cell -> Test -> Dataset` workflow.


In [ ]:
cell_type = derive_cell_type(specification)
workspace.add(cell_type)

cell = workspace.cell(cell_type, serial_number='coin-demo-001', source_type='lab')
test = workspace.test(
    cell,
    kind='capacity_check',
    protocol='0.2 mA constant-current discharge',
    instrument='Arbin micro-battery tester',
    status='completed',
)
dataset = workspace.dataset(
    cell,
    title='Detailed coin-cell capacity dataset',
    description='Small dataset linked to the detailed coin-cell descriptor walkthrough.',
    test=test,
    path=dataset_file,
    license='CC-BY-4.0',
)

{'derived_cell_type_id': cell_type.id, 'cell_model': cell_type.model, 'dataset_title': dataset.name}


In [ ]:
save_result = workspace.save(validation_policy='strict')

{
    'cell_type_count': save_result['index']['cell_type_count'],
    'dataset_count': save_result['index']['dataset_count'],
}


## 4. Build a publication package that includes the descriptor


In [ ]:
publication_result = workspace.build_publication_package(dataset, cell_specification=specification)

{
    'publish_path': publication_result['publish_path'],
    'cell_type_id': publication_result['cell_type_id'],
    'dataset_id': publication_result['dataset_id'],
    'triple_count': publication_result['triple_count'],
}
